In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import time
import datetime

!pip install schedule
import schedule

# Definir o tamanho do dataset
n_empresas = 1000
n_dias = 1

# Gerar datas
datas = pd.date_range(start='2022-01-01', periods=n_dias)

# Gerar características
np.random.seed(42)
data = {
    'ID_empresa': np.repeat(range(1, n_empresas + 1), n_dias),
    'Data': np.tile(datas, n_empresas),
    'Setor': np.random.choice(['Indústria', 'Comércio', 'Serviços'], n_empresas, p=[0.4, 0.3, 0.3]).repeat(n_dias),
    'Tamanho': np.random.choice(['Pequena', 'Média', 'Grande'], n_empresas, p=[0.5, 0.3, 0.2]).repeat(n_dias),
    'Receita': np.random.normal(100000, 50000, n_empresas * n_dias),
    'Custo': np.random.normal(50000, 20000, n_empresas * n_dias),
    'Dívida': np.random.exponential(200000, n_empresas * n_dias),
    'Histórico_pagamento': np.random.choice([0, 1, 2], n_empresas * n_dias, p=[0.2, 0.5, 0.3]) # 0: ruim, 1: médio, 2: bom
}

# Criar DataFrame
df = pd.DataFrame(data)

# Adicionar ruído aos dados
df['Receita'] += np.random.normal(0, 5000, n_empresas * n_dias)
df['Custo'] += np.random.normal(0, 2000, n_empresas * n_dias)
df['Dívida'] += np.random.normal(0, 10000, n_empresas * n_dias)

# Calcular indicadores financeiros
df['Margem_lucro'] = (df['Receita'] - df['Custo']) / df['Receita']
df['Índice_endividamento'] = df['Dívida'] / df['Receita']

# Selecionar as características relevantes
X = df[['Receita', 'Custo', 'Dívida', 'Histórico_pagamento', 'Margem_lucro', 'Índice_endividamento']]

# Normalizar os dados
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(X)

# Treinar o modelo K-Means
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto') # Adicionado random_state e n_init para reprodutibilidade e evitar warning
kmeans.fit(X_normalizado)

# Obter os grupos
grupos = kmeans.labels_

# Adicionar os grupos ao DataFrame
df['Grupo'] = grupos

# Calcular o risco para cada grupo
df['Risco'] = np.where(df['Grupo'] == 0, 'Alto',
                        np.where(df['Grupo'] == 1, 'Médio',
                                 np.where(df['Grupo'] == 2, 'Baixo',
                                          np.where(df['Grupo'] == 3, 'Normal', 'Muito Baixo'))))

def ingestao_diaria(data):
    # Gerar novas linhas com os dados do dia
    novas_linhas = pd.DataFrame({
        'ID_empresa': np.random.choice(range(1, n_empresas + 1), 10),
        'Data': [data]*10,
        'Setor': np.random.choice(['Indústria', 'Comércio', 'Serviços'], 10, p=[0.4, 0.3, 0.3]),
        'Tamanho': np.random.choice(['Pequena', 'Média', 'Grande'], 10, p=[0.5, 0.3, 0.2]),
        'Receita': np.random.normal(100000, 50000, 10),
        'Custo': np.random.normal(50000, 20000, 10),
        'Dívida': np.random.exponential(200000, 10),
        'Histórico_pagamento': np.random.choice([0, 1, 2], 10, p=[0.2, 0.5, 0.3])
    })

    # Adicionar ruído aos dados
    novas_linhas['Receita'] += np.random.normal(0, 5000, 10)
    novas_linhas['Custo'] += np.random.normal(0, 2000, 10)
    novas_linhas['Dívida'] += np.random.normal(0, 10000, 10)

    # Calcular indicadores financeiros
    novas_linhas['Margem_lucro'] = (novas_linhas['Receita'] - novas_linhas['Custo']) / novas_linhas['Receita']
    novas_linhas['Índice_endividamento'] = novas_linhas['Dívida'] / novas_linhas['Receita']

    # Selecionar as características relevantes
    X_novas = novas_linhas[['Receita', 'Custo', 'Dívida', 'Histórico_pagamento', 'Margem_lucro', 'Índice_endividamento']]

    # Normalizar os dados
    X_novas_normalizado = scaler.transform(X_novas)

    # Prever os grupos
    grupos_novas = kmeans.predict(X_novas_normalizado)

    # Adicionar os grupos às novas linhas
    novas_linhas['Grupo'] = grupos_novas

    # Calcular o risco para cada grupo
    novas_linhas['Risco'] = np.where(novas_linhas['Grupo'] == 0, 'Alto',
                                    np.where(novas_linhas['Grupo'] == 1, 'Médio',
                                             np.where(novas_linhas['Grupo'] == 2, 'Baixo',
                                                      np.where(novas_linhas['Grupo'] == 3, 'Normal', 'Muito Baixo'))))

    # Adicionar as novas linhas ao DataFrame
    global df
    df = pd.concat([df, novas_linhas])

    # Gerar alerta diário
    print(f"Alerta diário - {data}")
    print(novas_linhas[['ID_empresa', 'Data', 'Risco']])
    print()

# Gerar dados para 10 dias
data_atual = pd.to_datetime('2022-01-01')
for i in range(10):
    ingestao_diaria(data_atual)
    data_atual += pd.DateOffset(days=1)
    time.sleep(1)

Alerta diário - 2022-01-01 00:00:00
   ID_empresa       Data        Risco
0         406 2022-01-01       Normal
1         614 2022-01-01  Muito Baixo
2         775 2022-01-01  Muito Baixo
3         271 2022-01-01         Alto
4         736 2022-01-01        Baixo
5         651 2022-01-01         Alto
6         792 2022-01-01         Alto
7         511 2022-01-01        Médio
8         963 2022-01-01  Muito Baixo
9         286 2022-01-01        Baixo

Alerta diário - 2022-01-02 00:00:00
   ID_empresa       Data        Risco
0         955 2022-01-02         Alto
1         241 2022-01-02        Baixo
2         930 2022-01-02        Médio
3         843 2022-01-02        Baixo
4         351 2022-01-02       Normal
5         722 2022-01-02  Muito Baixo
6          23 2022-01-02         Alto
7         636 2022-01-02        Médio
8         598 2022-01-02        Baixo
9          43 2022-01-02  Muito Baixo

Alerta diário - 2022-01-03 00:00:00
   ID_empresa       Data        Risco
0         924 20